In [3]:
import os
import random
import logging
import gc
import pickle

import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
torch.set_num_threads(4)

from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH = 128
SEED = 42

TRAIN_PATH = "jigsaw-toxic-comment-train.csv"
VAL_PATH = "validation-processed-seqlen128.csv"
MAX_SAMPLES = 1000
OUTPUT_DIR = "./output"


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def load_data(path: str):
    df = pd.read_csv(path, on_bad_lines="skip", engine="python", encoding="utf-8")
    df["comment_text"] = df["comment_text"].fillna("")
    return df


def make_binary_label(df: pd.DataFrame):
    toxic_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
    if all(c in df.columns for c in toxic_cols):
        return (df[toxic_cols].sum(axis=1) > 0).astype(int).tolist()
    if "toxic" in df.columns:
        return df["toxic"].astype(int).tolist()
    raise ValueError("No toxic labels found")


def get_mbert_embeddings(texts, tokenizer, mbert, batch_size=64, device="cpu"):
    mbert.eval()
    mbert.to(device)
    all_embeddings = []
    logger.info("Extracting embeddings for %d texts...", len(texts))
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding", unit="batch"):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(batch_texts, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)
        with torch.no_grad():
            outputs = mbert(input_ids=input_ids, attention_mask=attention_mask)
            cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.extend(cls)
        gc.collect()
    return np.array(all_embeddings)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    set_seed(SEED)

    train_df = load_data(TRAIN_PATH)
    val_df = load_data(VAL_PATH)
    if MAX_SAMPLES is not None:
        train_df = train_df.sample(n=MAX_SAMPLES, random_state=SEED)

    train_texts = train_df["comment_text"].astype(str).tolist()
    train_labels = make_binary_label(train_df)
    val_texts = val_df["comment_text"].astype(str).tolist()
    val_labels = make_binary_label(val_df)

    logger.info("Train: %d, Val: %d, Toxic ratio: %.3f", len(train_texts), len(val_texts), sum(train_labels)/len(train_labels))

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    mbert = AutoModel.from_pretrained(MODEL_NAME)
    device = torch.device("cpu")

    cache_path = os.path.join(OUTPUT_DIR, "svm_embeddings.pkl")
    if os.path.exists(cache_path):
        logger.info("Loading cached embeddings...")
        with open(cache_path, "rb") as f:
            train_emb, val_emb = pickle.load(f)
    else:
        logger.info("Extracting mBERT embeddings...")
        train_emb = get_mbert_embeddings(train_texts, tokenizer, mbert, device=device)
        val_emb = get_mbert_embeddings(val_texts, tokenizer, mbert, device=device)
        with open(cache_path, "wb") as f:
            pickle.dump((train_emb, val_emb), f)
        logger.info("Cached to %s", cache_path)

    del mbert, tokenizer
    gc.collect()

    logger.info("Training SVM...")
    clf = SGDClassifier(loss="hinge", max_iter=1000, tol=1e-3, random_state=SEED, class_weight="balanced")
    clf.fit(train_emb, train_labels)
    predictions = clf.predict(val_emb)

    acc = accuracy_score(val_labels, predictions)
    f1 = f1_score(val_labels, predictions, pos_label=1)
    logger.info("SVM accuracy: %.4f, f1: %.4f", acc, f1)
    logger.info("Report:\n%s", classification_report(val_labels, predictions, digits=4))


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
